# Network Calibration

Loads the Douglas-Peucker network produced by `network_extraction.ipynb` and applies:

1. **Port exclusion** (`EXCLUDED_PORTS_BY_PORTID`) — remove unwanted ports entirely
2. **Manual port addition** (`MANUAL_PORTS_BY_PORTID`) — add ports from the IMF dataset that were not selected by the coverage threshold, connected to their nearest existing network nodes
3. **Manual edge additions** (`MANUAL_EDGES`) — add shortcuts or missing connections
4. **Manual port isolation** (`PORTS_WITH_MANUAL_ONLY_CONNECTIONS`) — strip all organic connections from selected ports, keeping only their manual edges
5. **Choke-point isolation** (`CHOKE_POINTS_WITH_MANUAL_ONLY_CONNECTIONS`) — strip all organic connections from selected choke points, keeping only their manual edges
6. **A\* cleanup** — re-runs A\* across all port/choke-point pairs on the now-modified graph and retains only nodes and edges that lie on at least one shortest path, removing any orphans left by the manual changes

**Run this notebook as often as needed** — it is fast (small graph) and overwrites the output files in-place.

In [1]:
import pickle
import os
import networkx as nx
import numpy as np
from math import radians, cos, sin, asin, sqrt
from tqdm import tqdm

print('Libraries loaded.')

Libraries loaded.


In [2]:
# ===========================================================================
# CONFIGURATION
# Edit the lists below, then re-run all cells to apply and export the
# calibrated network.
# ===========================================================================

# Paths (relative to this notebook)
NETWORK_PICKLE_IN   = 'network_outputs/network_dp.gpickle'          # input: raw D-P graph
NETWORK_PICKLE_OUT  = 'network_outputs/network_calibrated.gpickle'  # output: calibrated graph
NETWORK_GRAPHML_OUT = 'network_outputs/network_calibrated.graphml'  # output: calibrated GraphML

# ---------------------------------------------------------------------------
# Ports to exclude entirely (removed before any manual edges or A* cleanup).
# List of IMF portid values (as stored in the 'portid' column of port_data_imf.csv).
# ---------------------------------------------------------------------------
EXCLUDED_PORTS_BY_PORTID = ['port1330', #Turkmenbashi Port
                            'port100', #Baku
                            'port16' #Aktau
                            ]


# ---------------------------------------------------------------------------
# Manual ports to add from IMF port data (portids not captured by the
# coverage threshold in network_extraction.ipynb).
# Each port is connected to its MANUAL_PORT_NETWORK_CONNECTIONS nearest
# existing network nodes automatically. Additional specific connections can
# be added via MANUAL_EDGES below.
# ---------------------------------------------------------------------------
MANUAL_PORTS_BY_PORTID = []

# Number of nearest existing network nodes to connect each manually added port to.
MANUAL_PORT_NETWORK_CONNECTIONS = 5

# Data files needed to look up manually added port attributes.
IMF_PORT_DATA_FILE = '../../data/port_data_imf.csv'
BACI_CODES_FILE    = '../../data/raw_trade_data/BACI_country_codes.csv'

# ---------------------------------------------------------------------------
# Manual edges to add between ports (by IMF portid) and/or choke points
# (by name string). Each entry: (from_portid, to_portid_or_chokepoint_name).
# Edges are added in both directions with a Haversine distance weight.
# ---------------------------------------------------------------------------
MANUAL_EDGES = [('port1259', 'port754'), #Tampa-Mobile
                ('port1259', 'port86'), #Tampa-Habana
                ('port244', 'Yucatan Channel'), #Cienfuegos-Yucatan Channel
                ('port244', 'port389'), #Cienfuegos-Georgetown
                ('port142', 'port148'), #Belize City-Big Creek
                ('port142', 'Yucatan Channel'), #Belize City-Yucatan Channel
                ('port142', 'port389'), #Belize City-Georgetown
                ('port148', 'port1159'), #Big Creek-Puerto Barrios
                ('port148', 'port389'), #Big Creek-Georgetown
                ('port1159', 'port1036'), #Puerto Barrios-Puerto Cortes
                ('port1159', 'port389'), #Puerto Barrios-Georgetown
                ('port1036', 'port389'), #Puerto Cortes-Georgetown
                ('port1036', 'Yucatan Channel'), #Puerto Cortes-Yucatan Channel
                ('port240', 'port1052'), #Chirqui Grande-Puerto Moin
                ('port240', 'port700'), #Chirqui Grande-Manzanillo
                ('port934', 'port2330'), #Port Au Prince-Port Rhoades
                ('port1042', 'port51'), #Haina-Caucedo
                ('port1042', 'port871'), #Haina-Oranjestad
                ('fso167', 'Torres Strait'), #Timor-Leste Offshore Terminal-Torres Strait
                ('fso167', 'port955'), #Timor-Leste Offshore Terminal-Port Hedland
                ('port970', 'Torres Strait'), #Port Moresby-Torres Strait
                ('port747', 'port744'), #Mina Saqr-Jebel Ali
                ('port747', 'port106'), #Mina Saqr-Bandar Shahid Rajaee
                ('port654', 'port141'), #Liverpool-Belfast
                ('port654', 'port307'), #Liverpool-Dublin
                ('port2238', 'port1279'), #Hound Point-Teesport
                ('port2238', 'fso147'), #Hound Point-Norway Offshore Terminal 2
                ('port670', 'port379'), #Lulea-Gavle
                ('port670', 'port586'), #Lulea-Kokkola
                ('port586', 'port379'), #Kokkola-Gavle
                ('port586', 'port698'), #Kokkola-Pori
                ('port1', 'port354'), #Aabenraa-Fredericia
                ('port1', 'port2125'), #Aabenraa-Stigsnaes
                ('port446', 'port1396'), #Hamburg-Wilhelmshaven
                ('port766', 'port1075'), #Montreal-Quebec
                ('port1075', 'port938'), #Quebec-Port Cartier
                ('port938', 'port1181'), #Port Cartier-Sept Iles
                ('port1181', 'port2101'), #Sept Iles-Point Tupper
                ('port1181', 'port256'), #Sept Iles-Whiffen Head
                ('port1098', 'port256'), #Sept Iles-Whiffen Head
                ('port230', 'port1032'), #Charco Azul-Puerto Caldera
                ('port230', 'port2200'), #Charco Azul-Taboguilla
                ('port389', 'port1052'), #Georgetown-Puerto Moin
                ('port700', 'port1052'), #Manzanillo-Puerto Moin
                ('port389', 'port700'), #Georgetown-Manzanillo
                ('port700', 'port218'), #Manzanillo-Cartagena
                ('port101', 'port2200'), #Balboa-Taboguilla
                ('port183', 'port2200'), #Buena Ventura-Taboguilla
                ('fso167', 'Lombok Strait'), #Timor-Leste Offshore Terminal-Torres Strait
                ('fso167', 'port245'), #Timor-Leste Offshore Terminal-Port Hedland
                ('port1059', 'port1108'), #San Martin-Rosario
                ('port1108', 'port1145'), #Rosario-San Nicolas
                ('port1145', 'port210'), #San Nicolas-Campana
                ('port210', 'port184'), #Campana-Buenos Aires
                ('port153', 'port729'), #Bluff Harbour-Melbourne
                ('port153', 'port161'), #Bluff Harbour-Port Botany
                ('port966', 'port1388'), #Littleton-Wellington
                ('port966', 'port1388'), #Littleton-Bluff Harbour
                ('port1393', 'port1240'), #Marsden Point-Suva
                ('port1393', 'port636'), #Marsden Point-Lautoka
                ('port980', 'port636'), #Port Vila-Lautoka
                ('port619', 'port2200'), #La Libertad-Taboguilla
                ('port949', 'port700'), #Port Esquivel-Manzanillo
                ('Panama Canal', 'port700'), #Panama Canal-Manzanillo
                ('Panama Canal', 'port101'), #Panama Canal-Balboa
                ('Balabac Strait', 'port774'), #Balabac Strait-Muara
                ('Mindoro Strait', 'port125'), #Mindoro Strait-Batangas
                ('Mindoro Strait', 'port694'), #Mindoro Strait-Manila
                ('Mindoro Strait', 'port1232') #Mindoro Strait-Subic Bay
                ]

# ---------------------------------------------------------------------------
# Ports whose organic G_dp edges are stripped, keeping only those in
# MANUAL_EDGES. Ports with degree 0 after stripping are removed entirely.
# List of IMF portid values.
# ---------------------------------------------------------------------------
PORTS_WITH_MANUAL_ONLY_CONNECTIONS = ['port1259', #Tampa
                                      'port244', #Cienfuegos
                                      'port142', #Belize City
                                      'port148', #Big Creek
                                      'port1159', #Puerto Barrios
                                      'port1036', #Puerto Cortes
                                      'port240', #Chirqui Grande
                                      'port934', #Port Au Prince
                                      'port1042', #Haina
                                      'fso167', #Timor-Leste Offshore Terminal
                                      'port654', #Liverpool
                                      'port2238', #Hound Point
                                      'port670', #Lulea
                                      'port586', #Kokkola
                                      'port1', #Aabenraa
                                      'port446', #Hamburg
                                      'port766', #Montreal
                                      'port1075', #Quebec
                                      'port938', #Port Cartier
                                      'port1181', #Sept Iles
                                      'port230', #Charco Azul
                                      'port1059', #San Martin
                                      'port1108', #Rosario
                                      'port1145', #San Nicolas
                                      'port210', #Campana
                                      'port153', #Bluff Harbour
                                      'port966', #Littleton
                                      'port1052' #Puerto Moin
                                      ]

# ---------------------------------------------------------------------------
# Choke points whose organic G_dp edges are stripped, keeping only those in
# MANUAL_EDGES. Use this to fix incorrect auto-connections for a choke point.
# List of choke-point name strings (as in maritime_chokepoints.csv / node 'name').
# ---------------------------------------------------------------------------
CHOKE_POINTS_WITH_MANUAL_ONLY_CONNECTIONS = ['Panama Canal']

print('Config loaded.')

Config loaded.


In [3]:
# Load the Douglas-Peucker network saved by network_extraction.ipynb
with open(NETWORK_PICKLE_IN, 'rb') as f:
    G_dp = pickle.load(f)

print(f'Loaded G_dp: {G_dp.number_of_nodes():,} nodes, {G_dp.number_of_edges():,} edges')

port_nodes       = [n for n in G_dp.nodes() if G_dp.nodes[n].get('source') == 'port']
choke_nodes      = [n for n in G_dp.nodes() if G_dp.nodes[n].get('source') == 'choke_point']
print(f'  Ports:        {len(port_nodes)}')
print(f'  Choke points: {len(choke_nodes)}')

Loaded G_dp: 9,207 nodes, 17,563 edges
  Ports:        599
  Choke points: 28


In [4]:
# Build lookup tables from node attributes stored in G_dp
# (portid was added to node attrs by network_extraction.ipynb)
port_id_to_node = {
    G_dp.nodes[n]['portid']: n
    for n in port_nodes
    if 'portid' in G_dp.nodes[n]
}

choke_name_to_node = {
    G_dp.nodes[n]['name']: n
    for n in choke_nodes
}

print(f'Port lookup:        {len(port_id_to_node)} entries')
print(f'Choke-point lookup: {len(choke_name_to_node)} entries')

Port lookup:        599 entries
Choke-point lookup: 28 entries


In [5]:
# ===========================================================================
# STEP 0 — Port exclusion
# Remove unwanted ports from G_dp entirely, before any manual edges are added
# or A* cleanup runs. Excluded ports will not appear in the calibrated network.
# ===========================================================================
print('=' * 60)
print('STEP 0: Port exclusion')
print('=' * 60)

if not EXCLUDED_PORTS_BY_PORTID:
    print('EXCLUDED_PORTS_BY_PORTID is empty — nothing to do.')
else:
    removed = 0
    for portid in EXCLUDED_PORTS_BY_PORTID:
        node_id = port_id_to_node.get(portid)
        if node_id is None:
            print(f'  WARNING: Port {portid} not found in network — skipping')
            continue
        port_name = G_dp.nodes[node_id].get('portname', str(portid))
        G_dp.remove_node(node_id)
        del port_id_to_node[portid]  # keep lookup consistent
        print(f'  Removed: {port_name} (portid {portid})')
        removed += 1

    print(f'\nRemoved {removed} port(s).')
    print(f'G_dp now: {G_dp.number_of_nodes():,} nodes, {G_dp.number_of_edges():,} edges')

STEP 0: Port exclusion
  Removed: Turkmenbashi Port (portid port1330)
  Removed: Baku Port (portid port100)
  Removed: Aktau Port (portid port16)

Removed 3 port(s).
G_dp now: 9,204 nodes, 17,551 edges


In [6]:
def haversine_km(lon1, lat1, lon2, lat2):
    """Great-circle distance in km between two (lon, lat) points."""
    lon1, lat1, lon2, lat2 = map(radians, [lon1, lat1, lon2, lat2])
    dlon, dlat = lon2 - lon1, lat2 - lat1
    a = sin(dlat/2)**2 + cos(lat1) * cos(lat2) * sin(dlon/2)**2
    return 2 * asin(sqrt(a)) * 6371

def heuristic(n1, n2, G):
    return haversine_km(
        G.nodes[n1]['lon'], G.nodes[n1]['lat'],
        G.nodes[n2]['lon'], G.nodes[n2]['lat'],
    )

In [7]:
# ===========================================================================
# STEP 0.5 — Add manual ports from IMF data
# Ports listed in MANUAL_PORTS_BY_PORTID are loaded from the IMF port
# dataset and injected into G_dp as port nodes, connected to their
# MANUAL_PORT_NETWORK_CONNECTIONS nearest existing network nodes.
# They then participate in MANUAL_EDGES (Step 1) and A* cleanup (Step 3)
# just like any other port.
# ===========================================================================
import pandas as pd

print('=' * 60)
print('STEP 0.5: Adding manual ports')
print('=' * 60)

if not MANUAL_PORTS_BY_PORTID:
    print('MANUAL_PORTS_BY_PORTID is empty — nothing to add.')
else:
    imf  = pd.read_csv(IMF_PORT_DATA_FILE)
    baci = pd.read_csv(BACI_CODES_FILE)
    iso3_to_baci = dict(zip(baci['iso_3digit_alpha'], baci['country_name_abbreviation']))
    iso3_to_baci['TWN'] = 'Other Asia, nes'  # Taiwan uses S19 in BACI
    imf['baci_name'] = imf['ISO3'].map(iso3_to_baci)

    # Pre-compute coordinates of all existing nodes for nearest-neighbour search
    existing_node_ids = list(G_dp.nodes())
    node_lons = np.array([G_dp.nodes[n]['lon'] for n in existing_node_ids])
    node_lats = np.array([G_dp.nodes[n]['lat'] for n in existing_node_ids])

    ports_added = 0
    for portid in MANUAL_PORTS_BY_PORTID:
        if portid in port_id_to_node:
            print(f'  WARNING: {portid} already in network — skipping')
            continue

        rows = imf[imf['portid'] == portid]
        if rows.empty:
            print(f'  WARNING: {portid} not found in IMF data — skipping')
            continue
        row = rows.iloc[0]

        port_node_id = f'port_{portid}'
        G_dp.add_node(
            port_node_id,
            pos=(float(row['lon']), float(row['lat'])),
            lon=float(row['lon']),
            lat=float(row['lat']),
            source='port',
            portid=portid,
            portname=row['portname'],
            ISO3=row['ISO3'],
            country=str(row.get('baci_name', '') or ''),
        )

        # Find K nearest existing nodes and connect
        distances_km = np.array([
            haversine_km(float(row['lon']), float(row['lat']), nlon, nlat)
            for nlon, nlat in zip(node_lons, node_lats)
        ])
        k = min(MANUAL_PORT_NETWORK_CONNECTIONS, len(existing_node_ids))
        nearest_idxs = np.argsort(distances_km)[:k]

        for idx in nearest_idxs:
            neighbor = existing_node_ids[idx]
            dist_km  = float(distances_km[idx])
            G_dp.add_edge(port_node_id, neighbor,
                          distance=dist_km, length=dist_km,
                          source='manual_port_connection')

        port_id_to_node[portid] = port_node_id
        print(f'  Added: {row["portname"]} ({portid}), ISO3={row["ISO3"]}, '
              f'connected to {k} nearest nodes')
        ports_added += 1

    print(f'\nAdded {ports_added} manual port(s).')
    print(f'G_dp now: {G_dp.number_of_nodes():,} nodes, {G_dp.number_of_edges():,} edges')

STEP 0.5: Adding manual ports
MANUAL_PORTS_BY_PORTID is empty — nothing to add.


In [8]:
# ===========================================================================
# STEP 1 — Add manual edges
# ===========================================================================
print('=' * 60)
print('STEP 1: Adding manual edges')
print('=' * 60)

if not MANUAL_EDGES:
    print('MANUAL_EDGES is empty — nothing to add.')
else:
    edges_added = 0
    for from_id, to_target in MANUAL_EDGES:
        from_node = port_id_to_node.get(from_id)
        if from_node is None:
            print(f'  WARNING: Port {from_id} not found — skipping')
            continue

        # Try portid first, then fall back to choke-point name
        to_node = port_id_to_node.get(to_target)
        if to_node is None:
            to_node = choke_name_to_node.get(to_target)
        if to_node is None:
            print(f'  WARNING: "{to_target}" not found as portid or choke point — skipping')
            continue

        if G_dp.has_edge(from_node, to_node):
            continue  # already present

        dist = haversine_km(
            G_dp.nodes[from_node]['lon'], G_dp.nodes[from_node]['lat'],
            G_dp.nodes[to_node]['lon'],   G_dp.nodes[to_node]['lat'],
        )
        G_dp.add_edge(from_node, to_node, distance=dist, length=dist, source='manual')
        edges_added += 1

    print(f'Added {edges_added} manual edge(s).')
    print(f'G_dp now: {G_dp.number_of_nodes():,} nodes, {G_dp.number_of_edges():,} edges')

STEP 1: Adding manual edges
Added 62 manual edge(s).
G_dp now: 9,204 nodes, 17,613 edges


In [9]:
# ===========================================================================
# STEP 2 — Manual port isolation
# Strip all non-manual edges from selected ports.
# ===========================================================================
print('=' * 60)
print('STEP 2: Manual port isolation')
print('=' * 60)

if not PORTS_WITH_MANUAL_ONLY_CONNECTIONS:
    print('PORTS_WITH_MANUAL_ONLY_CONNECTIONS is empty — nothing to do.')
else:
    print(f'G_dp before: {G_dp.number_of_nodes():,} nodes, {G_dp.number_of_edges():,} edges')

    # Resolve MANUAL_EDGES to frozensets of node-ID pairs (for fast lookup)
    manual_node_pairs = set()
    for from_id, to_target in MANUAL_EDGES:
        from_node = port_id_to_node.get(from_id)
        if from_node is None:
            continue
        to_node = port_id_to_node.get(to_target)
        if to_node is None:
            to_node = choke_name_to_node.get(to_target)
        if to_node is None:
            continue
        manual_node_pairs.add(frozenset([from_node, to_node]))

    isolated_count = 0
    removed_count  = 0

    for portid in PORTS_WITH_MANUAL_ONLY_CONNECTIONS:
        node_id = port_id_to_node.get(portid)
        if node_id is None:
            print(f'  WARNING: Port {portid} not in lookup — skipping')
            continue
        if node_id not in G_dp:
            print(f'  WARNING: Port {portid} ({node_id}) not in G_dp — skipping')
            continue

        port_name = G_dp.nodes[node_id].get('portname', str(portid))
        edges_to_remove = [
            nbr for nbr in list(G_dp.neighbors(node_id))
            if frozenset([node_id, nbr]) not in manual_node_pairs
        ]
        for nbr in edges_to_remove:
            G_dp.remove_edge(node_id, nbr)

        remaining = G_dp.degree(node_id)
        if remaining == 0:
            G_dp.remove_node(node_id)
            print(f'  {port_name} ({portid}): removed {len(edges_to_remove)} edge(s) → node removed (degree 0)')
            removed_count += 1
        else:
            print(f'  {port_name} ({portid}): removed {len(edges_to_remove)} edge(s), kept {remaining} manual connection(s)')
            isolated_count += 1

    print(f'\nResult: {isolated_count} port(s) isolated, {removed_count} port(s) removed.')
    print(f'G_dp after:  {G_dp.number_of_nodes():,} nodes, {G_dp.number_of_edges():,} edges')

STEP 2: Manual port isolation
G_dp before: 9,204 nodes, 17,613 edges
  Tampa (port1259): removed 5 edge(s), kept 2 manual connection(s)
  Cienfuegos (port244): removed 4 edge(s), kept 2 manual connection(s)
  Belize City (port142): removed 5 edge(s), kept 3 manual connection(s)
  Big Creek (port148): removed 5 edge(s), kept 3 manual connection(s)
  Puerto Barrios (port1159): removed 5 edge(s), kept 3 manual connection(s)
  Puerto Cortes (port1036): removed 5 edge(s), kept 3 manual connection(s)
  Chiriqui Grande (port240): removed 4 edge(s), kept 2 manual connection(s)
  Port Au Prince (port934): removed 4 edge(s), kept 1 manual connection(s)
  Haina (port1042): removed 5 edge(s), kept 2 manual connection(s)
  Timor-Leste - Offshore Oil Terminal 1 (fso167): removed 1 edge(s), kept 4 manual connection(s)
  Liverpool (port654): removed 5 edge(s), kept 2 manual connection(s)
  Hound Point (port2238): removed 5 edge(s), kept 2 manual connection(s)
  Lulea (port670): removed 2 edge(s), kept

In [10]:
# ===========================================================================
# STEP 2b — Choke-point isolation
# Strip all non-manual edges from selected choke points, keeping only those
# added via MANUAL_EDGES. Choke points with degree 0 after stripping are
# removed entirely.
# ===========================================================================
print('=' * 60)
print('STEP 2b: Choke-point isolation')
print('=' * 60)

if not CHOKE_POINTS_WITH_MANUAL_ONLY_CONNECTIONS:
    print('CHOKE_POINTS_WITH_MANUAL_ONLY_CONNECTIONS is empty — nothing to do.')
else:
    print(f'G_dp before: {G_dp.number_of_nodes():,} nodes, {G_dp.number_of_edges():,} edges')

    # Resolve MANUAL_EDGES to frozensets of node-ID pairs (for fast lookup)
    # (may already exist from Step 2, but recompute to be safe)
    manual_node_pairs_choke = set()
    for from_id, to_target in MANUAL_EDGES:
        from_node = port_id_to_node.get(from_id)
        if from_node is None:
            from_node = choke_name_to_node.get(from_id)
        to_node = port_id_to_node.get(to_target)
        if to_node is None:
            to_node = choke_name_to_node.get(to_target)
        if from_node is None or to_node is None:
            continue
        manual_node_pairs_choke.add(frozenset([from_node, to_node]))

    isolated_count = 0
    removed_count  = 0

    for choke_name in CHOKE_POINTS_WITH_MANUAL_ONLY_CONNECTIONS:
        node_id = choke_name_to_node.get(choke_name)
        if node_id is None:
            print(f'  WARNING: Choke point "{choke_name}" not found in lookup — skipping')
            continue
        if node_id not in G_dp:
            print(f'  WARNING: Choke point "{choke_name}" ({node_id}) not in G_dp — skipping')
            continue

        edges_to_remove = [
            nbr for nbr in list(G_dp.neighbors(node_id))
            if frozenset([node_id, nbr]) not in manual_node_pairs_choke
        ]
        for nbr in edges_to_remove:
            G_dp.remove_edge(node_id, nbr)

        remaining = G_dp.degree(node_id)
        if remaining == 0:
            G_dp.remove_node(node_id)
            print(f'  "{choke_name}": removed {len(edges_to_remove)} edge(s) → node removed (degree 0)')
            removed_count += 1
        else:
            print(f'  "{choke_name}": removed {len(edges_to_remove)} edge(s), kept {remaining} manual connection(s)')
            isolated_count += 1

    print(f'\nResult: {isolated_count} choke point(s) isolated, {removed_count} choke point(s) removed.')
    print(f'G_dp after:  {G_dp.number_of_nodes():,} nodes, {G_dp.number_of_edges():,} edges')


STEP 2b: Choke-point isolation
G_dp before: 9,204 nodes, 17,496 edges
  "Panama Canal": removed 13 edge(s), kept 2 manual connection(s)

Result: 1 choke point(s) isolated, 0 choke point(s) removed.
G_dp after:  9,204 nodes, 17,483 edges


In [11]:
# ===========================================================================
# STEP 3 — A* cleanup
# Re-run A* across all port/choke-point pairs on the modified G_dp.
# Only nodes and edges that lie on at least one shortest path are kept.
# This removes any orphaned nodes or edges left by manual changes.
# ===========================================================================
print('=' * 60)
print('STEP 3: A* cleanup')
print('=' * 60)

# Refresh port and choke lists (some may have been removed in step 2)
port_nodes_now  = [n for n in G_dp.nodes() if G_dp.nodes[n].get('source') == 'port']
choke_nodes_now = [n for n in G_dp.nodes() if G_dp.nodes[n].get('source') == 'choke_point']
all_special = port_nodes_now + choke_nodes_now
n_special = len(all_special)

print(f'Special nodes: {len(port_nodes_now)} ports + {len(choke_nodes_now)} choke points = {n_special} total')
print(f'Total paths to compute: {n_special * (n_special - 1) // 2}')

used_nodes = set()
used_edges = set()
paths_found  = 0
paths_failed = 0

for i in tqdm(range(n_special), desc='A* cleanup'):
    src = all_special[i]
    for j in range(i + 1, n_special):
        tgt = all_special[j]
        try:
            path = nx.astar_path(
                G_dp, src, tgt,
                heuristic=lambda n1, n2: heuristic(n1, n2, G_dp),
                weight='distance',
            )
            used_nodes.update(path)
            for k in range(len(path) - 1):
                used_edges.add(frozenset([path[k], path[k+1]]))
            paths_found += 1
        except (nx.NetworkXNoPath, nx.NodeNotFound):
            paths_failed += 1

print(f'\nPaths found:  {paths_found}')
print(f'Paths failed: {paths_failed}')
print(f'Nodes on paths: {len(used_nodes)}')
print(f'Edges on paths: {len(used_edges)}')

STEP 3: A* cleanup
Special nodes: 596 ports + 28 choke points = 624 total
Total paths to compute: 194376


A* cleanup:   0%|          | 0/624 [00:00<?, ?it/s]

A* cleanup:   0%|          | 1/624 [00:01<20:15,  1.95s/it]

A* cleanup:   0%|          | 2/624 [00:03<17:54,  1.73s/it]

A* cleanup:   0%|          | 3/624 [00:06<21:44,  2.10s/it]

...

A* cleanup:  98%|█████████▊| 610/624 [11:29<00:00, 17.33it/s]

A* cleanup:  98%|█████████▊| 613/624 [11:30<00:00, 19.42it/s]

A* cleanup:  99%|█████████▊| 616/624 [11:30<00:00, 21.62it/s]

A* cleanup: 100%|██████████| 624/624 [11:30<00:00,  1.11s/it]


Paths found:  193753
Paths failed: 623
Nodes on paths: 8726
Edges on paths: 14634


In [12]:
# Build the calibrated graph from only used nodes and edges
G_calibrated = nx.Graph()

for node in used_nodes:
    G_calibrated.add_node(node, **G_dp.nodes[node])

for edge in used_edges:
    n1, n2 = tuple(edge)
    G_calibrated.add_edge(n1, n2, **G_dp.edges[n1, n2])

nodes_removed = G_dp.number_of_nodes() - G_calibrated.number_of_nodes()
edges_removed = G_dp.number_of_edges() - G_calibrated.number_of_edges()

print(f'G_calibrated: {G_calibrated.number_of_nodes():,} nodes, {G_calibrated.number_of_edges():,} edges')
print(f'Pruned:       {nodes_removed} node(s), {edges_removed} edge(s) removed by A* cleanup')

G_calibrated: 8,726 nodes, 14,634 edges
Pruned:       478 node(s), 2849 edge(s) removed by A* cleanup


In [13]:
# Export — overwrites the files produced by network_extraction.ipynb
print('=' * 60)
print('EXPORTING CALIBRATED NETWORK')
print('=' * 60)

# Pickle
with open(NETWORK_PICKLE_OUT, 'wb') as f:
    pickle.dump(G_calibrated, f)
print(f'Saved pickle:  {NETWORK_PICKLE_OUT}')

# GraphML (can't store tuples — convert pos to string)
G_export = G_calibrated.copy()
for node in G_export.nodes():
    if 'pos' in G_export.nodes[node]:
        pos = G_export.nodes[node]['pos']
        G_export.nodes[node]['pos_str'] = f'{pos[0]},{pos[1]}'
        del G_export.nodes[node]['pos']

nx.write_graphml(G_export, NETWORK_GRAPHML_OUT)
print(f'Saved GraphML: {NETWORK_GRAPHML_OUT}')

print(f'\nFinal network: {G_calibrated.number_of_nodes()} nodes, {G_calibrated.number_of_edges()} edges')

EXPORTING CALIBRATED NETWORK
Saved pickle:  network_outputs/network_calibrated.gpickle


Saved GraphML: network_outputs/network_calibrated.graphml

Final network: 8726 nodes, 14634 edges
